# Online Retail Customer & Sales Analytics

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

## Loading Dataset

In [ ]:
df = pd.read_csv(r"D:\Datasets\Online Retail data.csv")
df.head(5)

## Data Understanding

In [ ]:
print("Shape:", df.shape)
df.info()
df.describe()

## Checking Missing Values

In [ ]:
df.isnull().sum()

## Data Cleaning

In [ ]:
# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove negative/zero quantity (returns or errors)
df = df[df['Quantity'] > 0]

# Remove negative price
df = df[df['UnitPrice'] > 0]

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

## saving cleaned data
df.to_csv("cleaned_retail_data.csv", index=False)

## Create RFM Features

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalPrice': 'sum'}).reset_index()                      # Monetary

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

## Feature Engineering

In [ ]:
# Average Order Value
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

# Customer Lifetime
first_purchase = df.groupby('CustomerID')['InvoiceDate'].min()
last_purchase = df.groupby('CustomerID')['InvoiceDate'].max()

rfm['Customer_Lifetime'] = (last_purchase - first_purchase).dt.days.values

## Creating Target (Churn)

In [16]:
# Churn: no purchase in last 90 days
rfm['Churn'] = rfm['Recency'].apply(lambda x: 1 if x > 90 else 0)

# Check distribution
print(rfm['Churn'].value_counts())

Churn
0    2889
1    1449
Name: count, dtype: int64


## Prepare Data

In [17]:
X = rfm[['Recency', 'Frequency', 'Monetary', 'AOV', 'Customer_Lifetime']]
y = rfm['Churn']

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.2, random_state=42)

## Logistic Regression

In [19]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       561
           1       1.00      1.00      1.00       307

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868

ROC-AUC: 1.0


## Random Forest

In [21]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest ===")
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))


=== Random Forest ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       561
           1       1.00      1.00      1.00       307

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868

ROC-AUC: 1.0


## Feature Importance

In [22]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
print("\nFeature Importance:")
print(importance.sort_values(ascending=False))


Feature Importance:
Recency              0.873535
Customer_Lifetime    0.067541
Frequency            0.029078
Monetary             0.023865
AOV                  0.005982
dtype: float64


## Churn Probability

In [23]:
rfm['Churn_Probability'] = rf.predict_proba(X)[:, 1]

# High-risk customers
high_risk = rfm[rfm['Churn_Probability'] > 0.7]

print("\nHigh Risk Customers:")
print(high_risk.head())


High Risk Customers:
   CustomerID  Recency  Frequency  Monetary      AOV  Customer_Lifetime  \
0     12346.0      326          1   77183.6  77183.6                  0   
4     12350.0      310          1     334.4    334.4                  0   
6     12353.0      204          1      89.0     89.0                  0   
7     12354.0      232          1    1079.4   1079.4                  0   
8     12355.0      214          1     459.4    459.4                  0   

   Churn  Churn_Probability  
0      1               0.99  
4      1               1.00  
6      1               1.00  
7      1               1.00  
8      1               1.00  


## Saving output

In [26]:
rfm.to_csv("customer_churn_predictions.csv", index=False)

## Key Insights

- Total revenue reached ~8.6M across ~19K orders and ~4.3K customers, indicating strong sales volume  
- A small group of top customers contributes a significant share of total revenue, highlighting dependency on high-value users  
- Majority of customers are one-time buyers, indicating low retention and high churn risk  
- Customers with recency > 90 days and low frequency show the highest churn probability  
- Revenue shows clear seasonal spikes in specific months, indicating demand concentration  

## Conclusion

- RFM-based features (Recency, Frequency, Monetary) effectively capture customer behavior for churn prediction  
- The model identifies high-risk customers with measurable probability, enabling targeted retention strategies  
- Improving retention of even a small percentage of customers can significantly impact overall revenue (~8.6M baseline)  
- Integrating analytics and machine learning enables more accurate and proactive business decision-making  